<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/velocidad_vientoIDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Velocidad del viento — estaciones IDEAM cercanas a la laguna

Este notebook selecciona las estaciones más cercanas y construye tres series independientes de velocidad del viento. Los valores originales no se imputan, interpolan, suavizan, agregan ni eliminan por parecer atípicos.

La base original permanece en `df_raw`. En las copias de trabajo se conserva una sola aparición de cada fila completamente idéntica.

## 1. Librerías y configuración

Usamos `pandas`, `numpy` y `matplotlib`. No se requieren mapas web ni servicios externos.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
RUTA_CSV = Path("Velocidad_del_Viento_20260818_SOLO_BOLIVAR.csv")
FORMATO_FECHA = "%Y %b %d %I:%M:%S %p"
print("Archivo encontrado:", RUTA_CSV.exists())

**Interpretación.** `True` confirma que el CSV está disponible junto al notebook.

## 2. Carga de los datos originales

Los códigos se leen como texto para conservar sus ceros iniciales. `df_raw` no se sobrescribirá.

In [ ]:
df_raw = pd.read_csv(
    RUTA_CSV,
    dtype={"CodigoEstacion": "string", "CodigoSensor": "string"},
)
FILAS_ORIGINALES = len(df_raw)
COLUMNAS_ORIGINALES = df_raw.columns.tolist()
print(f"Dimensiones: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
display(df_raw.head(3))

**Interpretación.** La base contiene 3.554.573 filas y 12 columnas. Cada fila representa una medición asociada a una estación, sensor y fecha.

## 3. Exploración básica

Revisamos tipos, faltantes, fechas, rango, sensores y unidad antes de crear cualquier serie. Las conversiones se guardan en objetos auxiliares y no cambian `df_raw`.

In [ ]:
descripcion_columnas = pd.DataFrame({
    "columna": df_raw.columns,
    "tipo_cargado": df_raw.dtypes.astype(str).values,
    "faltantes": df_raw.isna().sum().values,
})
display(descripcion_columnas)

fechas_aux = pd.to_datetime(
    df_raw["FechaObservacion"], format=FORMATO_FECHA, errors="coerce"
)
valores_aux = pd.to_numeric(df_raw["ValorObservado"], errors="coerce")
print("Periodo:", fechas_aux.min(), "a", fechas_aux.max())
print("Fechas no convertibles:", fechas_aux.isna().sum())
print("Valores no convertibles:", valores_aux.isna().sum())
print("Rango observado:", valores_aux.min(), "a", valores_aux.max(), "m/s")
print("Estaciones:", df_raw["CodigoEstacion"].nunique())
for columna in ["CodigoSensor", "DescripcionSensor", "UnidadMedida"]:
    print(f"{columna}:", sorted(df_raw[columna].dropna().astype(str).str.strip().unique()))

**Interpretación.** El periodo disponible va del 30-ene-2012 al 17-ago-2026. No hay celdas vacías ni errores de conversión. Los valores están entre 0 y 79,2 m/s y aparecen los sensores `0103` y `0111`.

## 4. Duplicados y valores que requieren revisión

Una copia exacta debe coincidir en las 12 columnas originales. También contamos ceros, negativos y velocidades elevadas. Estos conteos son diagnósticos y no filtros.

In [ ]:
mascara_duplicados = df_raw.duplicated(keep=False)
n_copias_exactas = int(df_raw.duplicated().sum())
grupos_duplicados = (
    df_raw.loc[mascara_duplicados]
    .groupby(COLUMNAS_ORIGINALES, dropna=False).size()
    .sort_values(ascending=False)
)
print(f"Copias exactas adicionales: {n_copias_exactas:,}")
print(f"Grupos de filas repetidas: {len(grupos_duplicados):,}")
print("Mayor número de apariciones de una misma fila:", grupos_duplicados.max())
print(f"Valores negativos: {(valores_aux < 0).sum():,}")
print(f"Valores iguales a cero: {(valores_aux == 0).sum():,}")
print(f"Valores mayores de 30 m/s: {(valores_aux > 30).sum():,}")
print(f"Valores mayores de 50 m/s: {(valores_aux > 50).sum():,}")
display(grupos_duplicados.head(5).rename("numero_de_apariciones").reset_index())

**Interpretación.** Hay 257.728 copias adicionales realmente idénticas. No existen velocidades negativas. Los ceros pueden representar calma y se conservan; las velocidades elevadas se revisarán sin eliminarlas.

## 5. Catálogo de estaciones

Construimos una fila por código. Se muestra el nombre más frecuente y la coordenada más reciente, además de advertir cuántas coordenadas diferentes aparecen.

In [ ]:
base_catalogo = df_raw[["CodigoEstacion", "NombreEstacion", "Municipio",
                        "Latitud", "Longitud", "FechaObservacion"]].copy()
base_catalogo["Fecha"] = fechas_aux
nombres = base_catalogo.groupby("CodigoEstacion")["NombreEstacion"].agg(
    lambda s: s.astype(str).str.strip().value_counts().index[0]
).rename("NombreEstacion")
municipios = base_catalogo.groupby("CodigoEstacion")["Municipio"].agg(
    lambda s: s.astype(str).str.strip().value_counts().index[0]
).rename("Municipio")
resumen = base_catalogo.groupby("CodigoEstacion").agg(
    registros=("Fecha", "size"), inicio=("Fecha", "min"), fin=("Fecha", "max"),
    coordenadas_distintas=("Latitud", lambda s: len(set(zip(
        s.round(8), base_catalogo.loc[s.index, "Longitud"].round(8)
    )))),
)
ultima_coordenada = (base_catalogo.sort_values("Fecha")
    .drop_duplicates("CodigoEstacion", keep="last")
    .set_index("CodigoEstacion")[["Latitud", "Longitud"]])
catalogo = pd.concat([nombres, municipios, ultima_coordenada, resumen], axis=1).reset_index()
display(catalogo.sort_values("registros", ascending=False))

**Interpretación.** Hay 12 códigos de estación. Las coordenadas múltiples se hacen visibles para evitar representar silenciosamente como fija una ubicación que cambió en el archivo.

## 6. Región de interés y distancia Haversine

Usamos el mismo polígono de la laguna. Su centroide sirve como punto de referencia y Haversine calcula la distancia sobre una Tierra esférica.

In [ ]:
ROI_COORDS = [
    (-75.476052, 10.517524), (-75.476117, 10.518747),
    (-75.473158, 10.519223), (-75.470516, 10.525108),
    (-75.469572, 10.524876), (-75.471686, 10.518916),
    (-75.468394, 10.517219), (-75.468952, 10.516459),
]

def centroide_poligono(coordenadas):
    area_doble = suma_x = suma_y = 0.0
    for (x0, y0), (x1, y1) in zip(coordenadas, coordenadas[1:] + coordenadas[:1]):
        cruz = x0 * y1 - x1 * y0
        area_doble += cruz
        suma_x += (x0 + x1) * cruz
        suma_y += (y0 + y1) * cruz
    return suma_y / (3 * area_doble), suma_x / (3 * area_doble)

def haversine_km(latitud, longitud, latitud_roi, longitud_roi):
    radio = 6371.0088
    lat1, lon1 = np.radians(np.asarray(latitud, float)), np.radians(np.asarray(longitud, float))
    lat2, lon2 = math.radians(latitud_roi), math.radians(longitud_roi)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * math.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radio * np.arcsin(np.sqrt(a))

LAT_ROI, LON_ROI = centroide_poligono(ROI_COORDS)
catalogo["distancia_km"] = haversine_km(
    catalogo["Latitud"], catalogo["Longitud"], LAT_ROI, LON_ROI
).round(2)
catalogo_distancias = catalogo.sort_values("distancia_km").reset_index(drop=True)
print(f"Centroide: {LAT_ROI:.6f}, {LON_ROI:.6f}")
display(catalogo_distancias[["CodigoEstacion", "NombreEstacion", "Municipio",
    "Latitud", "Longitud", "distancia_km", "registros", "inicio", "fin"]])

**Interpretación.** El código vigente de Rafael Núñez está a 9,35 km y UNAD a 13,99 km. El código histórico del aeropuerto aparece entre ambos por corresponder prácticamente al mismo sitio, pero no se concatena.

## 7. Selección y mapa

Seleccionamos dos sitios físicos independientes y mostramos su posición respecto a la laguna.

In [ ]:
CODIGOS_SELECCIONADOS = ["0014015080", "1206500136"]
seleccion_estaciones = (catalogo_distancias[
    catalogo_distancias["CodigoEstacion"].isin(CODIGOS_SELECCIONADOS)]
    .set_index("CodigoEstacion").loc[CODIGOS_SELECCIONADOS].reset_index())
display(seleccion_estaciones[["CodigoEstacion", "NombreEstacion",
                               "Latitud", "Longitud", "distancia_km"]])

fig, ax = plt.subplots(figsize=(8, 6))
lon_roi = [p[0] for p in ROI_COORDS] + [ROI_COORDS[0][0]]
lat_roi = [p[1] for p in ROI_COORDS] + [ROI_COORDS[0][1]]
ax.fill(lon_roi, lat_roi, color="#63c7da", alpha=0.45, label="Laguna (ROI)")
ax.scatter(LON_ROI, LAT_ROI, color="navy", marker="x", s=70, label="Centroide")
colores_estacion = ["#d1495b", "#2a9d8f"]
for (_, fila), color in zip(seleccion_estaciones.iterrows(), colores_estacion):
    ax.scatter(fila["Longitud"], fila["Latitud"], s=75, color=color)
    ax.plot([LON_ROI, fila["Longitud"]], [LAT_ROI, fila["Latitud"]],
            linestyle="--", linewidth=1, color=color)
    texto = f'{fila["NombreEstacion"]}\n{fila["CodigoEstacion"]} — {fila["distancia_km"]:.2f} km'
    ax.annotate(texto, (fila["Longitud"], fila["Latitud"]),
                xytext=(7, 4), textcoords="offset points", fontsize=8)
ax.set(xlabel="Longitud", ylabel="Latitud",
       title="Laguna y estaciones de velocidad del viento")
ax.set_aspect(1 / math.cos(math.radians(LAT_ROI)))
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación.** Las dos estaciones están al suroeste de la laguna. Su cercanía respalda la selección, pero no garantiza que reproduzcan exactamente el viento sobre el agua.

## 8. Copia de trabajo y tres series independientes

Retiramos solamente copias idénticas. Luego separamos Rafael `0103`, Rafael `0111` y UNAD `0103`; ninguna medición se mezcla entre sensores.

In [ ]:
seleccion_raw = df_raw[df_raw["CodigoEstacion"].isin(CODIGOS_SELECCIONADOS)].copy()
seleccion_raw["es_duplicado_exacto"] = seleccion_raw.duplicated(subset=COLUMNAS_ORIGINALES)
duplicados_por_serie = (
    seleccion_raw.groupby(["CodigoEstacion", "CodigoSensor"])["es_duplicado_exacto"]
    .sum().astype(int).rename("copias_excluidas").reset_index()
)
serie_limpia = seleccion_raw.drop_duplicates(subset=COLUMNAS_ORIGINALES).copy()
serie_limpia["FechaObservacion"] = pd.to_datetime(
    serie_limpia["FechaObservacion"], format=FORMATO_FECHA, errors="coerce")
serie_limpia["ValorObservado"] = pd.to_numeric(serie_limpia["ValorObservado"], errors="coerce")
serie_limpia = serie_limpia.sort_values(
    ["CodigoEstacion", "CodigoSensor", "FechaObservacion"]).reset_index(drop=True)

rafael_0103 = serie_limpia[(serie_limpia["CodigoEstacion"] == "0014015080") &
                            (serie_limpia["CodigoSensor"] == "0103")].copy()
rafael_0111 = serie_limpia[(serie_limpia["CodigoEstacion"] == "0014015080") &
                            (serie_limpia["CodigoSensor"] == "0111")].copy()
unad_0103 = serie_limpia[(serie_limpia["CodigoEstacion"] == "1206500136") &
                          (serie_limpia["CodigoSensor"] == "0103")].copy()

series = {
    "Rafael Núñez — 0103 (10 min)": (rafael_0103, 10),
    "Rafael Núñez — 0111 (2 min)": (rafael_0111, 2),
    "UNAD — 0103 (10 min)": (unad_0103, 10),
}
display(duplicados_por_serie)
for nombre, (datos, _) in series.items():
    print(f"{nombre}: {len(datos):,} registros")
print("df_raw continúa intacto:", len(df_raw) == FILAS_ORIGINALES and
      df_raw.columns.tolist() == COLUMNAS_ORIGINALES)

**Interpretación.** Quedan 246.913 registros de Rafael `0103`, 502.985 de Rafael `0111` y 208.044 de UNAD `0103`. Los tres objetos tienen la misma importancia, pero permanecen independientes.

In [ ]:
# Verificación: después de retirar copias exactas no debe repetirse una fecha dentro de una serie.
revision_claves = []
for nombre, (datos, _) in series.items():
    repetidas = datos.duplicated(["CodigoEstacion", "CodigoSensor", "FechaObservacion"]).sum()
    conflictos = (datos.groupby("FechaObservacion")["ValorObservado"].nunique() > 1).sum()
    revision_claves.append({"serie": nombre, "fechas_repetidas": repetidas,
                            "conflictos_dentro_del_sensor": conflictos})
display(pd.DataFrame(revision_claves))

**Interpretación.** Los resultados deben ser cero: las filas descartadas sí eran copias completas y no mediciones diferentes del mismo sensor.

## 9. Coincidencias entre sensores y valores elevados

Los sensores del aeropuerto pueden medir en el mismo instante y producir valores diferentes. Esto no se considera automáticamente un error. También localizamos velocidades mayores de 30 m/s sin retirarlas.

In [ ]:
aeropuerto = serie_limpia[serie_limpia["CodigoEstacion"] == "0014015080"]
comparacion_sensores = aeropuerto.groupby("FechaObservacion").agg(
    sensores=("CodigoSensor", "nunique"), valores=("ValorObservado", "nunique"))
tiempos_compartidos = comparacion_sensores[comparacion_sensores["sensores"] > 1]
print(f"Tiempos compartidos: {len(tiempos_compartidos):,}")
print(f"Con valores diferentes: {(tiempos_compartidos['valores'] > 1).sum():,}")
print(f"Con el mismo valor: {(tiempos_compartidos['valores'] == 1).sum():,}")

elevados = serie_limpia[serie_limpia["ValorObservado"] > 30][
    ["CodigoEstacion", "CodigoSensor", "FechaObservacion", "ValorObservado"]]
print(f"Observaciones seleccionadas mayores de 30 m/s: {len(elevados):,}")
display(elevados.sort_values("ValorObservado", ascending=False))

**Interpretación.** Los sensores coinciden temporalmente 100.164 veces y difieren en 93.134. Rafael tiene 46 observaciones superiores a 30 m/s, con máximos de 71 y 79 m/s; se marcan para revisión, no se corrigen.

## 10. Cobertura y periodos sin observaciones

Cada serie usa su propia frecuencia esperada. Las rejillas temporales se crean solamente para contar ausencias y nunca se unen a los valores observados.

In [ ]:
filas_cobertura = []
faltantes_por_serie = {}
huecos_por_serie = {}
for nombre, (datos, paso_min) in series.items():
    datos = datos.sort_values("FechaObservacion")
    diferencias = datos["FechaObservacion"].diff().dt.total_seconds().div(60)
    inicio, fin = datos["FechaObservacion"].min(), datos["FechaObservacion"].max()
    rejilla = pd.date_range(inicio, fin, freq=f"{paso_min}min")
    observados = pd.DatetimeIndex(datos["FechaObservacion"])
    faltantes = rejilla.difference(observados)
    faltantes_por_serie[nombre] = faltantes
    huecos = pd.DataFrame({"fin_del_hueco": datos["FechaObservacion"],
                           "diferencia_min": diferencias})
    huecos = huecos[huecos["diferencia_min"] > paso_min].copy()
    huecos["inicio_del_hueco"] = huecos["fin_del_hueco"] - pd.to_timedelta(
        huecos["diferencia_min"], unit="min")
    huecos_por_serie[nombre] = huecos.nlargest(10, "diferencia_min")
    filas_cobertura.append({
        "serie": nombre, "inicio": inicio, "fin": fin,
        "observaciones": len(datos), "paso_esperado_min": paso_min,
        "paso_mediano_min": diferencias.median(), "tiempos_teoricos": len(rejilla),
        "tiempos_ausentes": len(faltantes),
        "cobertura_pct": round(100 * len(observados) / len(rejilla), 2),
        "mayor_hueco_dias": round(diferencias.max() / 1440, 2),
    })
cobertura = pd.DataFrame(filas_cobertura)
display(cobertura)
for nombre, faltantes in faltantes_por_serie.items():
    print(f"{nombre}: primeros tiempos ausentes:", faltantes[:5].tolist())

**Interpretación.** Las coberturas aproximadas son 67,88 %, 65,63 % y 55,95 %. Los tiempos ausentes continúan siendo ausencias; no se convierten en ceros ni en valores interpolados.

In [ ]:
for nombre, huecos in huecos_por_serie.items():
    print("\n", nombre)
    display(huecos[["inicio_del_hueco", "fin_del_hueco", "diferencia_min"]])

conteos_anuales = []
for nombre, (datos, _) in series.items():
    for anio, cantidad in datos.groupby(datos["FechaObservacion"].dt.year).size().items():
        conteos_anuales.append({"serie": nombre, "anio": anio, "observaciones": cantidad})
tabla_anual = pd.DataFrame(conteos_anuales).pivot(
    index="anio", columns="serie", values="observaciones").fillna(0).astype(int)
display(tabla_anual)

**Interpretación.** UNAD no tiene observaciones durante 2023. El sensor `0111` comienza en septiembre de 2023; los ceros anteriores de la tabla son ausencia de registros, no velocidad nula.

## 11. Series originales: rango completo

Los tres paneles usan el mismo eje de 0–80 m/s. Así se conservan y comparan todos los valores, incluidos los elevados. Los puntos rojos señalan observaciones mayores de 30 m/s.

In [ ]:
PALETA = ["#d1495b", "#6a4c93", "#2a9d8f"]
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True, sharey=True)
for ax, (nombre, (datos, _)), color in zip(axes, series.items(), PALETA):
    ax.plot(datos["FechaObservacion"], datos["ValorObservado"],
            linestyle="None", marker=",", color=color, alpha=0.65)
    altos = datos[datos["ValorObservado"] > 30]
    ax.scatter(altos["FechaObservacion"], altos["ValorObservado"],
               s=18, color="red", label="> 30 m/s", zorder=3)
    ax.set_title(nombre, loc="left", fontsize=10)
    ax.set_ylabel("m/s")
    ax.set_ylim(0, 80)
    ax.grid(alpha=0.2)
    if len(altos): ax.legend(loc="upper left")
axes[-1].set_xlabel("Fecha de observación")
fig.suptitle("Velocidad del viento — rango completo de las tres series")
plt.tight_layout()
plt.show()

**Interpretación.** La escala común permite comprobar que las velocidades elevadas solo aparecen en Rafael Núñez. También hace que el comportamiento habitual, concentrado por debajo de 10 m/s, se vea comprimido.

## 12. Vista ampliada del comportamiento habitual

Mostramos exactamente las mismas series con el eje limitado visualmente a 0–15 m/s. No se filtra el DataFrame: se informa cuántas observaciones quedan fuera de la ventana.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True, sharey=True)
for ax, (nombre, (datos, _)), color in zip(axes, series.items(), PALETA):
    ax.plot(datos["FechaObservacion"], datos["ValorObservado"],
            linestyle="None", marker=",", color=color, alpha=0.65)
    fuera = int((datos["ValorObservado"] > 15).sum())
    ax.set_title(nombre, loc="left", fontsize=10)
    ax.text(0.99, 0.93, f"{fuera} valores sobre 15 m/s", transform=ax.transAxes,
            ha="right", va="top", fontsize=9,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor=color))
    ax.set_ylabel("m/s")
    ax.set_ylim(0, 15)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("Fecha de observación")
fig.suptitle("Velocidad del viento — vista ampliada de 0 a 15 m/s")
plt.tight_layout()
plt.show()

**Interpretación.** Quedan fuera de esta vista 11 valores de Rafael `0103`, 58 de Rafael `0111` y ninguno de UNAD. Siguen presentes en los DataFrames y en la gráfica completa.

## 13. Estadísticas básicas

Calculamos las estadísticas por separado. La mediana, cuartiles y percentil 99 ayudan a entender por qué unos pocos valores muy altos no representan el comportamiento habitual.

In [ ]:
filas_estadisticas = []
for nombre, (datos, _) in series.items():
    v = datos["ValorObservado"].dropna()
    filas_estadisticas.append({
        "serie": nombre, "n": len(v), "media": v.mean(),
        "mediana": v.median(), "desviacion_estandar": v.std(),
        "minimo": v.min(), "q25": v.quantile(0.25),
        "q75": v.quantile(0.75), "p99": v.quantile(0.99),
        "maximo": v.max(), "mayores_30": int((v > 30).sum()),
    })
estadisticas = pd.DataFrame(filas_estadisticas).round(2)
display(estadisticas)

**Interpretación.** Los percentiles 99 de Rafael permanecen por debajo de 9 m/s pese a sus máximos de 71 y 79 m/s. Esto confirma que los valores elevados son infrecuentes, pero no demuestra por sí solo que sean errores.

## 14. Conclusiones y limitaciones

- Las dos ubicaciones físicas más cercanas son Rafael Núñez y UNAD.
- Los tres flujos estación–sensor se analizan con igual importancia, pero nunca se mezclan.
- Los ceros, huecos y velocidades elevadas permanecen visibles y sin corregir.
- UNAD no tiene datos durante 2023 y `0111` comienza en septiembre de ese año.
- El entorno aeroportuario y urbano puede diferir del microclima sobre la laguna.
- Las fechas no tienen zona horaria declarada.
- Cualquier unión, agregación, imputación o tratamiento de valores elevados requiere una justificación y decisión posterior.